In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/rBMRB')
print("working folder:", os.getcwd())


# Groups of the training set for K-fold

Groups the training entries of `cluster_30p` by the same two criteria as the split in `Screening_7_match.ipynb`. Each group stays whole within one fold, and no training data is lost, because linked entries are merged instead of removed.

- Foldseek: members of one cluster of `cluster_30p_old/result_cluster.tsv`.
- Sequence homology: the search of the split (`cluster_30p/leak_filter/all_vs_all.m8`) with identity ≥ 0.3, alignment covering ≥ 80% of either sequence, E ≤ 1e-3; or the same BMRB accession.

The groups are the connected components of these links.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd

BASE       = Path("../rBMRB")
ROOT       = BASE / "cluster_30p"
FS_CLUSTER = BASE / "cluster_30p_old" / "result_cluster.tsv"      # Foldseek clustering used for the split
TRAIN_LIST = ROOT / "split_output_filtered" / "train_entry_ids.txt"
TRAIN_DIR  = ROOT / "structures_split_filtered" / "train"
ALL_VS_ALL = ROOT / "leak_filter" / "all_vs_all.m8"
CRITERIA   = ROOT / "split_output_filtered" / "criteria.json"
OUT        = ROOT / "structures_split_filtered" / "train_cluster"
OUT.mkdir(parents=True, exist_ok=True)

# Same homology criteria as the split (read back from criteria.json to keep them in sync)
crit = json.loads(CRITERIA.read_text())["homology"]
MIN_ID, MIN_COV, MAX_EVALUE = crit["min_identity"], crit["min_coverage_either"], crit["max_evalue"]
print(f"criteria: identity >= {MIN_ID}, either-coverage >= {MIN_COV}, E <= {MAX_EVALUE}")

entries = sorted(l.strip() for l in open(TRAIN_LIST) if l.strip())
entry_set = set(entries)
pdbs = {p.stem for p in TRAIN_DIR.glob("*.pdb")}
assert pdbs == entry_set, "train structure dir does not match train_entry_ids.txt"
print(f"{len(entries)} train entries")

## 1. Foldseek clusters, restricted to train

In [ ]:
def bare(name):
    """File name -> entry id (drop .pdb and the test_ prefix used in the split structure dir)."""
    return name.removesuffix(".pdb").removeprefix("test_")

fs = pd.read_csv(FS_CLUSTER, sep="\t", names=["rep", "member"], dtype=str)
fs["rep"], fs["member"] = fs["rep"].map(bare), fs["member"].map(bare)
missing = entry_set - set(fs["member"])
assert not missing, f"{len(missing)} train entries not in the Foldseek clustering, e.g. {sorted(missing)[:5]}"
fs = fs[fs["member"].isin(entry_set)]
print(f"Foldseek clusters containing train entries: {fs['rep'].nunique()}")

fs_pairs = [(g[0], m) for g in fs.groupby("rep")["member"].apply(list) for m in g[1:]]

## 2. Homologous and same-accession pairs

In [ ]:
FMT = "query,target,fident,alnlen,qcov,tcov,evalue,bits"
aln = pd.read_csv(ALL_VS_ALL, sep="\t", names=FMT.split(","))
for c in ("query", "target"):
    aln[c] = aln[c].astype(str).str.removesuffix(".pdb")

seq_hits = aln[aln["query"].isin(entry_set) & aln["target"].isin(entry_set)
               & (aln["query"] != aln["target"])
               & (aln.fident >= MIN_ID)
               & (aln[["qcov", "tcov"]].max(axis=1) >= MIN_COV)
               & (aln.evalue <= MAX_EVALUE)]

acc = lambda e: e.split("_")[0]   # BMRB accession
by_acc = defaultdict(list)
for e in entries:
    by_acc[acc(e)].append(e)
acc_pairs = [(g[0], other) for g in by_acc.values() for other in g[1:]]

link_pairs = pd.concat([
    pd.DataFrame(fs_pairs, columns=["query", "target"]).assign(rule="foldseek_cluster"),
    seq_hits[["query", "target"]].assign(rule="seq"),
    pd.DataFrame(acc_pairs, columns=["query", "target"]).assign(rule="accession"),
], ignore_index=True)
print("linking pairs by rule:", link_pairs["rule"].value_counts().to_dict())

# Diagnostic: sequence-homologous pairs that the Foldseek clustering split apart
fs_rep = dict(zip(fs["member"], fs["rep"]))
n_cross = sum(fs_rep[q] != fs_rep[t] for q, t in zip(seq_hits["query"], seq_hits["target"]))
print(f"sequence-homologous pairs crossing Foldseek clusters (merged below): {n_cross}")

## 3. Connected components, written to `result_cluster.tsv`

In [ ]:
parent = {e: e for e in entries}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for q, t in zip(link_pairs["query"], link_pairs["target"]):
    union(q, t)

components = defaultdict(list)
for e in entries:
    components[find(e)].append(e)

# Representative: the member with the most distinct linked partners (ties broken by name)
partners = defaultdict(set)
for q, t in zip(link_pairs["query"], link_pairs["target"]):
    partners[q].add(t)
    partners[t].add(q)
rep_of = {}
for members in components.values():
    rep = max(members, key=lambda m: (len(partners[m]), m))
    for m in members:
        rep_of[m] = rep

rows = []
for rep in sorted(set(rep_of.values())):
    members = sorted(m for m, r in rep_of.items() if r == rep)
    members.remove(rep)
    rows += [(rep, rep)] + [(rep, m) for m in members]   # rep line first, foldseek convention
final = pd.DataFrame(rows, columns=["rep", "member"])
final.to_csv(OUT / "result_cluster.tsv", sep="\t", header=False, index=False)

sizes = final["rep"].value_counts()
print(f"groups: {len(sizes)} (Foldseek clusters in train: {fs['rep'].nunique()}) | singletons: {(sizes == 1).sum()}")
print(f"largest: {sizes.iloc[0]} entries ({sizes.iloc[0] / len(entries):.1%} of train); "
      f"largest 10: {sizes.head(10).tolist()}")
print(f"wrote {OUT / 'result_cluster.tsv'}")

## 4. Check

In [ ]:
check = pd.read_csv(OUT / "result_cluster.tsv", sep="\t", names=["rep", "member"], dtype=str)
assert check["member"].is_unique, "an entry appears in more than one group"
assert set(check["member"]) == entry_set, "cluster file does not cover exactly the train entries"
m2c = dict(zip(check["member"], check["rep"]))

for rule, sub in link_pairs.groupby("rule"):
    n = int((sub["query"].map(m2c) != sub["target"].map(m2c)).sum())
    print(f"{rule:17s}: {len(sub):6d} pairs, {n} span two groups")
    assert n == 0, f"{rule} pairs span two groups"
print("OK: K-fold splits that keep each group whole share no Foldseek cluster, "
      "sequence homolog or BMRB accession between folds")